## Data science _ hw2 _ Q1 _ part 2: 

### pre processing the raw csv file :

In [1]:
import pandas as pd

try:
    df = pd.read_csv('zomato.csv', encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv('zomato.csv', encoding='latin1')

#  Clean numeric columns (Rate and Cost)
df['rate'] = df['rate'].apply(lambda x: float(str(x).split('/')[0]) if (pd.notna(x) and '/' in str(x)) else None)
df['approx_cost(for two people)'] = pd.to_numeric(df['approx_cost(for two people)'].astype(str).str.replace(',', ''), errors='coerce')
df['votes'] = pd.to_numeric(df['votes'], errors='coerce').fillna(0).astype(int)

# Clean text fields and remove new lines
text_columns = ['address', 'name', 'phone', 'location', 'rest_type', 'cuisines', 'menu_item', 'listed_in(type)', 'listed_in(city)']
for col in text_columns:
    df[col] = df[col].astype(str).str.replace('\n', ' ').str.replace('\r', ' ').str.strip()

# Replace 'nan' values  with real None values for the database
df = df.replace('nan', None)


df = df.dropna(how='all')# Remove rows completely empty

# Remove rows with no restarant name or addres
df = df.dropna(subset=['name', 'address'])

df = df.reset_index(drop=True)# Reset the index 


print(f"Number of rows after removing empty records: {len(df)}")

# Display the result for all columns
print("--- Cleaned Data Overview ---")
print(df.head().T) 

Number of rows after removing empty records: 12429
--- Cleaned Data Overview ---
                                                                             0  \
address                      942, 21st Main Road, 2nd Stage, Banashankari, ...   
name                                                                     Jalsa   
online_order                                                               Yes   
book_table                                                                 Yes   
rate                                                                       4.1   
votes                                                                      775   
phone                                             080 42297555  +91 9743772233   
location                                                          Banashankari   
rest_type                                                        Casual Dining   
cuisines                                        North Indian, Mughlai, Chinese   
approx_cost(for t

### we notice a problem here in the adress content , we have invalid characters(mojibake): 

In [2]:
# find rows with adress length >255 char
long_addresses = df[df['address'].str.len() > 255]

print(f"Count of rows with unusual Address {len(long_addresses)}")
print("\n--- show the trouble rows---")

# show first 5 
for index, row in long_addresses.head(5).iterrows():
    print(f"Index: {index}")
    print(f"Name: {row['name']}")
    print(f"Address Length: {len(row['address'])}")
    print(f"Address Content: {row['address']}")
    print("-" * 30)

Count of rows with unusual Address 1

--- show the trouble rows---
Index: 9986
Name: Brownie Heaven
Address Length: 346
Address Content: Shop no. LG-5, Spendid Plaza, 5th ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂAÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ Block, 100 Feet Road, Koramangala 5th Block, Bangalore
------------------------------


In [3]:
import re

# Remove non-ASCII characters
def clean_mojibake(text):
    if not isinstance(text, str):
        return text
    
    cleaned = re.sub(r'[^\x00-\x7F]+', '', text)# Remove invalid characters
    
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()# Remove extra spaces
    
    return cleaned


df['address'] = df['address'].apply(clean_mojibake)# Clean address column

trouble_row = df.loc[9986]# Check problematic row (9986)

print("--- Fixed address for Koramangala branch (Row 9986) ---")
print(f"Name: {trouble_row['name']}")
print(f"Address: {trouble_row['address']}")


--- Fixed address for Koramangala branch (Row 9986) ---
Name: Brownie Heaven
Address: Shop no. LG-5, Spendid Plaza, 5th A Block, 100 Feet Road, Koramangala 5th Block, Bangalore


In [4]:
# find rows with adress length >255 char
long_addresses = df[df['address'].str.len() > 255]

print(f"Count of rows with unusual Address {len(long_addresses)}")
print("\n--- show the trouble rows---")

# show first 5 
for index, row in long_addresses.head(5).iterrows():
    print(f"Index: {index}")
    print(f"Name: {row['name']}")
    print(f"Address Length: {len(row['address'])}")
    print(f"Address Content: {row['address']}")
    print("-" * 30)

Count of rows with unusual Address 0

--- show the trouble rows---


In [5]:
# Save cleaned dataframe
# index=False -> do not save row index
# utf-8-sig -> opens correctly in Excel
#df.to_csv('zomato_cleaned.csv', index=False, encoding='utf-8-sig')

df.to_csv('zomato_cleaned.csv', index=False, encoding='utf-8')

print("the cleaned file has been saved with name 'zomato_cleaned.csv' .")

the cleaned file has been saved with name 'zomato_cleaned.csv' .
